# Real Pipeline Validation

Notebook enxuto para validar o funcionamento operacional do pipeline real.

## Papel deste notebook

Este notebook responde principalmente a pergunta:

- o pipeline real monta corretamente o tensor de entrada da rede neural a partir do dataset e do VCF reais?

Para auditoria semantica detalhada do tratamento de mutacoes/INDELs, use:

- `synthetic_indel_demo.ipynb`

## Contrato atual do pipeline

- unico config suportado: `one_gene_10_individuals.yaml`
- tensor por amostra: `(2, 4, L)`
- eixo 0: `H1`, `H2`
- eixo 1: `values`, `ins_mask`, `del_mask`, `valid_mask`
- tratamento de INDEL: sempre dinamico
- restricoes atuais:
  - 1 gene
  - 1 output
  - 1 track
  - `downsample_factor = 1`


In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != 'notebooks':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'genotype_based_predictor' / 'notebooks'
PACKAGE_DIR = NOTEBOOK_DIR.parent
REPO_ROOT = PACKAGE_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from genotype_based_predictor.config import load_config
from genotype_based_predictor.data_pipeline import prepare_data
from genotype_based_predictor.dynamic_indel_alignment import DynamicIndelAligner
from genotype_based_predictor.experiment import setup_experiment_dir
from genotype_based_predictor.genomic_dataset import GenomicDataset
from genotype_based_predictor.indel_tensor_builder import build_global_aligned_tensor
from genotype_based_predictor.models.cnn2_model import CNN2AncestryPredictor


In [2]:
CONFIG_PATH = PACKAGE_DIR / 'configs' / 'one_gene_10_individuals.yaml'
config = load_config(CONFIG_PATH)
display({
    'config_path': str(CONFIG_PATH),
    'dataset_dir': config.dataset_input.dataset_dir,
    'view_path': config.dataset_input.view_path,
    'tensor_layout': config.dataset_input.tensor_layout,
    'alphagenome_outputs': config.dataset_input.alphagenome_outputs,
    'haplotype_mode': config.dataset_input.haplotype_mode,
    'window_center_size': config.dataset_input.window_center_size,
    'downsample_factor': config.dataset_input.downsample_factor,
    'selected_track_index': config.dataset_input.selected_track_index,
    'genes_to_use': config.dataset_input.genes_to_use,
    'indel_include_valid_mask': config.dataset_input.indel_include_valid_mask,
})


{'config_path': '/home/breno/I2CA/genomics/genotype_based_predictor/configs/one_gene_10_individuals.yaml',
 'dataset_dir': '/dados/GENOMICS_DATA/v1/1kG_high_coverage',
 'view_path': '/home/breno/I2CA/genomics/genotype_based_predictor/views/one_gene_10_individuals.view.json',
 'tensor_layout': 'haplotype_channels',
 'alphagenome_outputs': ['rna_seq'],
 'haplotype_mode': 'H1+H2',
 'window_center_size': 32768,
 'downsample_factor': 1,
 'selected_track_index': 0,
 'genes_to_use': ['MC1R'],
 'indel_include_valid_mask': True}

## Etapa 1: Validacao End-To-End

Aqui verificamos que o pipeline monta batches no formato esperado e que a CNN2 consegue fazer forward pass.


In [3]:
experiment_dir = setup_experiment_dir(config, str(CONFIG_PATH))
full_ds, train_loader, val_loader, test_loader = prepare_data(config, experiment_dir)
batch = next(iter(train_loader))
features, targets = batch[:2]

display({
    'train_batch_features_shape': tuple(features.shape),
    'train_batch_targets_shape': tuple(targets.shape) if hasattr(targets, 'shape') else str(type(targets)),
    'single_sample_shape': tuple(features[0].shape),
    'expected_single_sample_shape': '(2, 4, L)',
    'input_shape_for_model_metadata': full_ds.get_input_shape(),
    'num_classes': full_ds.get_num_classes(),
})

model = CNN2AncestryPredictor(config, full_ds.get_input_shape(), full_ds.get_num_classes())
logits = model(features)
display({'model_output_shape': tuple(logits.shape)})
display(logits)


📁 Experimento: 
/dados/GENOMICS_DATA/v1/1kG_high_coverage_runs/cnn2_rna_seq_H1+H2_haplotype_channels_32768_log_s1k1x16f4_s2f8_s3f16
_gpavg_fc64_L64-32_relu_0.3_adam

   Nome: cnn2_rna_seq_H1+H2_haplotype_channels_32768_log_s1k1x16f4_s2f8_s3f16_gpavg_fc64_L64-32_relu_0.3_adam

✓ Cache válido e compatível

Cache encontrado: rna_seq_H1+H2_32768_ds1_log_shuf_view3dd85ebab914

Cache: 2026-05-03T19:08:12.925799 | 10 amostras

...

{'train_batch_features_shape': (1, 2, 4, 32768),
 'train_batch_targets_shape': (1,),
 'single_sample_shape': (2, 4, 32768),
 'expected_single_sample_shape': '(2, 4, L)',
 'input_shape_for_model_metadata': (8, 32768),
 'num_classes': 5}

Modelo CNN2 criado: 8×32768 → s1f4/s2f8/s3f16 → gpool → fc64 → 5

• Params: 2,077

{'model_output_shape': (1, 5)}

tensor([[ 0.2579, -0.2256, -0.2867, -0.0102,  0.0659]],
       grad_fn=<AddmmBackward0>)

## Etapa 2: Fonte de verdade das variantes

O pipeline real usa o VCF original da janela para decidir como expandir o eixo.


In [4]:
dataset = GenomicDataset(
    dataset_dir=Path(config.dataset_input.dataset_dir),
    load_predictions=True,
    load_sequences=False,
    cache_sequences=False,
)
with open(Path(config.dataset_input.view_path)) as f:
    view_payload = json.load(f)
sample_ids_path = Path(view_payload['sample_ids_path'])
sample_ids = [line.strip() for line in sample_ids_path.read_text().splitlines() if line.strip()]
gene_name = config.dataset_input.genes_to_use[0]

ref_window_dir = Path(config.dataset_input.dataset_dir) / 'references' / 'windows' / gene_name
with open(ref_window_dir / 'window_metadata.json') as f:
    window_metadata = json.load(f)
raw_variant_source = window_metadata['raw_variant_source']
region = f"{window_metadata['chromosome']}:{int(window_metadata['start']) + 1}-{int(window_metadata['end']) + 1}"

display({
    'gene_name': gene_name,
    'num_selected_samples': len(sample_ids),
    'region': region,
    'vcf_path': raw_variant_source['vcf_path'],
})


[INFO] GenomicLongevityDataset inicializado:
  • Dataset: non_longevous_dataset_genes
  • Indivíduos: 3202
  • Load predictions: True
  • Load sequences: False
  • Cache sequences: False


{'gene_name': 'MC1R',
 'num_selected_samples': 10,
 'region': 'chr16:89654404-90178691',
 'vcf_path': '/dados/GENOMICS_DATA/top3/longevity_dataset/vcf_chromosomes/1kGP_high_coverage_Illumina.chr16.filtered.SNV_INDEL_SV_phased_panel.vcf.gz'}

In [5]:
def query_window_variants(sample_id, region):
    fmt = '%CHROM\t%POS\t%REF\t%ALT[\t%GT]\n'
    cmd = ['bcftools', 'query', '-s', sample_id, '-r', region, '-f', fmt, raw_variant_source['vcf_path']]
    proc = subprocess.run(cmd, check=True, capture_output=True, text=True)
    rows = []
    for line in proc.stdout.splitlines():
        chrom, pos, ref, alt, gt = line.split('\t')
        rows.append({'sample_id': sample_id, 'chrom': chrom, 'pos_1based': int(pos), 'ref': ref, 'alt': alt, 'gt': gt})
    return pd.DataFrame(rows)

variant_preview_df = query_window_variants(sample_ids[0], region)
display(variant_preview_df.head(20))
print('Numero de variantes na janela para a primeira amostra:', len(variant_preview_df))


,sample_id,chrom,pos_1based,ref,alt,gt
0,HG00096,chr16,89654423,T,G,0|0
1,HG00096,chr16,89654429,C,T,0|0
2,HG00096,chr16,89654461,C,T,0|0
3,HG00096,chr16,89654501,T,A,1|1
4,HG00096,chr16,89654532,AGTCG,A,0|0
5,HG00096,chr16,89654536,G,A,0|0
6,HG00096,chr16,89654558,C,T,0|0
7,HG00096,chr16,89654615,G,A,0|0
8,HG00096,chr16,89654654,AT,A,0|0
9,HG00096,chr16,89654666,G,T,0|0


Numero de variantes na janela para a primeira amostra: 17564


## Etapa 3: Alinhamento dinamico real

O `DynamicIndelAligner` produz os artefatos que definem a montagem do tensor:

- `copy_from_indices`
- `expanded_indices`
- `insertion_indices`
- `deletion_indices`


In [6]:
aligner = DynamicIndelAligner(Path(config.dataset_input.dataset_dir), selected_sample_ids=sample_ids)
gene_mapping = aligner._load_gene_mapping(gene_name)
display({
    'expanded_length_full_window': aligner.get_expanded_length(gene_name),
    'num_samples_with_mapping': len(gene_mapping.get('samples', {})),
})

example_sample = sample_ids[0]
example_h1 = aligner.get_haplotype_entry(gene_name, example_sample, 'H1')
example_h2 = aligner.get_haplotype_entry(gene_name, example_sample, 'H2')
display({
    'example_sample': example_sample,
    'h1_copy_pairs': len(example_h1.get('expanded_indices', [])),
    'h1_insertion_positions': len(example_h1.get('insertion_indices', [])),
    'h1_deletion_positions': len(example_h1.get('deletion_indices', [])),
    'h2_copy_pairs': len(example_h2.get('expanded_indices', [])),
    'h2_insertion_positions': len(example_h2.get('insertion_indices', [])),
    'h2_deletion_positions': len(example_h2.get('deletion_indices', [])),
})

mapping_preview_df = pd.DataFrame({
    'copy_from_index': example_h1.get('copy_from_indices', [])[:40],
    'expanded_index': example_h1.get('expanded_indices', [])[:40],
})
display(mapping_preview_df)


{'expanded_length_full_window': 525001, 'num_samples_with_mapping': 10}

{'example_sample': 'HG00096',
 'h1_copy_pairs': 524307,
 'h1_insertion_positions': 136,
 'h1_deletion_positions': 117,
 'h2_copy_pairs': 524225,
 'h2_insertion_positions': 149,
 'h2_deletion_positions': 212}

,copy_from_index,expanded_index
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
5,5,5
6,6,6
7,7,7
8,8,8
9,9,9


## Etapa 4: Tensor global central real

A entrada da rede neural e construida apenas na janela central de interesse, mas em um eixo expandido minimo que cobre as mutacoes relevantes.


In [7]:
global_tensor, global_meta, global_rows = build_global_aligned_tensor(
    dataset=dataset,
    aligner=aligner,
    gene_name=gene_name,
    sample_ids=sample_ids,
    haplotypes=['H1', 'H2'],
    output_name=config.dataset_input.alphagenome_outputs[0],
    track_index=config.dataset_input.selected_track_index,
    neutral_value=config.dataset_input.indel_neutral_value,
    include_valid_mask=True,
    center_window_size=config.dataset_input.window_center_size,
)
display({
    'global_tensor_shape': tuple(global_tensor.shape),
    'center_slice': global_meta['center_slice'],
})
display(pd.DataFrame(global_rows).head(20))


{'global_tensor_shape': (10, 2, 4, 468120),
 'center_slice': {'center_ref_start': 245760,
  'center_ref_end': 278528,
  'center_expanded_start': 539,
  'center_expanded_end': 468659,
  'center_ref_length': 32768,
  'center_expanded_length': 468120}}

,sample_id,haplotype,copied_positions,insertion_positions,deletion_positions,valid_positions,tensor_shape
0,HG00096,H1,467426,136,117,467426,"(4, 468120)"
1,HG00096,H2,467344,149,212,467344,"(4, 468120)"
2,HG00097,H1,467240,225,392,467240,"(4, 468120)"
3,HG00097,H2,467354,149,202,467354,"(4, 468120)"
4,HG00099,H1,467444,218,181,467444,"(4, 468120)"
5,HG00099,H2,467312,158,253,467312,"(4, 468120)"
6,HG00100,H1,467313,189,283,467313,"(4, 468120)"
7,HG00100,H2,467418,138,127,467418,"(4, 468120)"
8,HG00101,H1,467420,235,222,467420,"(4, 468120)"
9,HG00101,H2,467250,168,325,467250,"(4, 468120)"


## O que este notebook nao tenta provar

Este notebook nao entra em auditoria mutacao-a-mutacao do caso real.

Para isso, use:

- `synthetic_indel_demo.ipynb`

Esse notebook sintetico e o local correto para verificar manualmente se:

- delecoes marcam `del_mask` nas colunas esperadas
- insercoes marcam `ins_mask` nas colunas esperadas
- `valid_mask` bate com as posicoes realmente copiadas
- o tensor final condiz com a expectativa humana em exemplos pequenos
